# Lesson 32: Retrieval-Augmented Generation (RAG) and its architecture 📚🤖
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain **why** RAG exists and when to use it instead of fine-tuning
2. Draw the **RAG architecture**: the ingestion pipeline and the query pipeline
3. Build RAG **from scratch**: load → chunk → embed → store → retrieve → augment → generate
4. Improve it with **hybrid search** and **re-ranking**, and **evaluate** it
5. Rebuild it in a few lines with **LangChain**

**⏱️ Timing:** why & architecture 10' | from scratch 20' | evaluation & upgrades 20' | LangChain 10'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 📖 An analogy: an open-book exam
An LLM on its own is a student sitting a **closed-book** exam: it can only use what it memorised during training. It may not know your school's rules, recent news or private documents — and when it doesn't know, it may make things up (Lesson 14).

**RAG** turns it into an **open-book** exam: before answering, we **look up** the relevant pages and hand them to the model.

| Problem | RAG helps because… |
|---|---|
| Knowledge cut-off | Documents can be updated any time, no retraining |
| Private data | Your files stay in your database |
| Hallucination | The model answers **from the sources** and can **cite** them |
| Cost | Much cheaper than fine-tuning for facts (Lesson 26) |

## 🏗️ RAG architecture
```
INGESTION (done once, or whenever documents change)
  documents ──► load ──► split into chunks ──► embed each chunk ──► store in vector DB
                                                                        │
QUERY (every question)                                                  ▼
  question ──► embed ──► retrieve top-k similar chunks ◄────────── vector DB
                                │
                                ▼
                  augment: prompt = instructions + chunks + question
                                │
                                ▼
                     generate with the LLM ──► answer + citations
```

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install openai sentence-transformers rank_bm25
import warnings, re, numpy as np; warnings.filterwarnings("ignore")
from openai import OpenAI
from sentence_transformers import SentenceTransformer
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def llm(prompt, temperature=0):
    r = client.chat.completions.create(model=MODEL, temperature=temperature, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

In [ ]:
HANDBOOK = {
"attendance.md": """# Attendance
Students must arrive by 8:40am. Students arriving after 8:40am must sign in at the main office and collect a late slip.
If a student is sick, a parent must phone the attendance line (09 555 0101) before 9:00am or send a message through the school app.
Students leaving early must bring a signed note and sign out at the office. Unexplained absences are followed up by the dean.""",
"devices.md": """# Devices and phones
Phones must be switched off and stored in lockers during class time. Teachers may allow devices for a specific learning activity.
Phones may be used at interval and lunch in outdoor areas only. Confiscated phones can be collected from the office at 3:15pm.
Every Year 9 and 10 student needs a laptop or Chromebook with at least 6 hours of battery. The school Wi-Fi network is called StudentNet.""",
"library.md": """# Library
The library is open 8:00am to 4:30pm on weekdays and is closed on weekends and public holidays.
Students may borrow up to 5 books for three weeks. Overdue books incur no fines, but borrowing is paused until books are returned.
Quiet study rooms can be booked at the front desk for groups of up to four students. Printing costs 10 cents per page.""",
"uniform.md": """# Uniform
The school jersey must be worn from May to September. Black leather shoes are required; sports shoes are only for PE.
Hats must be worn outdoors in Terms 1 and 4 (sun-smart policy). Jewellery is limited to a watch and one pair of small studs.
Uniform items can be bought from the uniform shop, open Tuesdays and Thursdays 8:00–9:00am.""",
"assessment.md": """# Assessment
Assessment extensions must be requested from the subject teacher at least two school days before the due date.
Students who are sick on an assessment day must provide a medical certificate within five days.
Using AI tools in an assessment is only allowed when the teacher's task sheet says so, and any AI use must be acknowledged.
Plagiarism, including copying AI-generated work without acknowledgement, results in the assessment not being awarded.""",
"sport.md": """# Sport and activities
Sports trials are held in the first two weeks of each term. The senior basketball team trains Mondays and Wednesdays 3:30–5pm.
Students need a signed permission slip and must pay the activity fee before attending any trip.
The swimming sports day is in week 5 of Term 1. Togs, a towel and sunscreen are required.""",
}
import os
os.makedirs("handbook", exist_ok=True)
for name, text in HANDBOOK.items():
    open(f"handbook/{name}", "w").write(text)
print("Wrote", len(HANDBOOK), "handbook files:", list(HANDBOOK))

## Part 1: Ingestion — chunk, embed, store 🧩
**Chunking** splits long documents into pieces small enough to be precise but big enough to make sense. Here: by sentence, grouped in twos, with **overlap** so ideas at the edges aren't cut in half.

In [ ]:
def chunk(text, size=2, overlap=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.split("\n", 1)[1].replace("\n", " ").strip())
    step = size - overlap
    return [" ".join(sentences[i:i + size]) for i in range(0, max(len(sentences) - overlap, 1), step)]

chunks, sources = [], []
for name, text in HANDBOOK.items():
    for c in chunk(text):
        chunks.append(c); sources.append(name)
vectors = embedder.encode(chunks, normalize_embeddings=True)
print(len(chunks), "chunks. Example:\n", sources[3], "→", chunks[3])

## Part 2: Query — retrieve, augment, generate 🔎

In [ ]:
def retrieve(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ q
    top = scores.argsort()[::-1][:k]
    return [(chunks[i], sources[i], float(scores[i])) for i in top]

def rag(question, k=3, show=False):
    hits = retrieve(question, k)
    context = "\n".join(f"[{n + 1}] ({src}) {text}" for n, (text, src, _) in enumerate(hits))
    prompt = f"""You are the school's help desk. Answer the question using ONLY the sources below.
Cite sources like [1]. If the sources don't contain the answer, say "I don't know — please ask the office."

Sources:
{context}

Question: {question}
Answer:"""
    if show: print(prompt, "\n" + "-" * 60)
    return llm(prompt)

print(rag("Can I use my mobile at lunchtime?", show=True))

In [ ]:
for q in ["What happens if I return a library book late?",
          "Am I allowed to use ChatGPT for my essay?",
          "Who is the school principal?"]:                 # not in the handbook!
    print(f"❓ {q}\n🤖 {rag(q)}\n")

### 🧩 Exercise 1
1. Compare with **no RAG**: `llm("Who is the school principal?")`. What does the plain model do?
2. Change `chunk(size=...)` to 1 and to 4. Re-run ingestion. Which size gives the best answers? (Too small → missing context; too big → irrelevant text and fewer chunks fit.)
3. Change `k` to 1 and to 6. What's the trade-off?

## Part 3: Evaluate retrieval 📏
If retrieval finds the wrong chunk, even a perfect LLM can't answer. A simple metric: **hit rate** = how often the right source appears in the top k.

In [ ]:
eval_set = [
    ("What time do I have to be at school?", "attendance.md"),
    ("Where do I pick up my confiscated phone?", "devices.md"),
    ("How much is printing?", "library.md"),
    ("Do I need a hat?", "uniform.md"),
    ("I was sick on test day, what do I need?", "assessment.md"),
    ("When does the basketball team practise?", "sport.md"),
    ("What do I bring to swimming sports?", "sport.md"),
    ("What is the name of the wifi?", "devices.md"),
    ("Can I wear earrings?", "uniform.md"),
    ("How do I get more time for an assignment?", "assessment.md"),
]
def hit_rate(search_fn, k=3):
    hits = sum(any(src == want for _, src, _ in search_fn(q, k)) for q, want in eval_set)
    return hits / len(eval_set)

for k in [1, 3]:
    print(f"Vector search hit rate @{k}: {hit_rate(retrieve, k):.0%}")

## Part 4: Upgrades used in real systems 🔧
### Hybrid search = keywords + meaning
**BM25** is an improved TF-IDF (Lesson 21) that's great at **exact words** (names, codes like "StudentNet"). Vector search is great at **meaning**. Hybrid search combines both rankings.

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([re.findall(r"\w+", c.lower()) for c in chunks])

def keyword_search(question, k=3):
    scores = bm25.get_scores(re.findall(r"\w+", question.lower()))
    return [(chunks[i], sources[i], float(scores[i])) for i in scores.argsort()[::-1][:k]]

def hybrid_search(question, k=3):
    # Reciprocal Rank Fusion: reward chunks that rank high in EITHER list
    fused = {}
    for results in [retrieve(question, 10), keyword_search(question, 10)]:
        for rank, (text, src, _) in enumerate(results):
            fused[(text, src)] = fused.get((text, src), 0) + 1 / (60 + rank)
    best = sorted(fused.items(), key=lambda x: -x[1])[:k]
    return [(t, s, score) for (t, s), score in best]

for name, fn in [("vector", retrieve), ("keyword (BM25)", keyword_search), ("hybrid", hybrid_search)]:
    print(f"{name:15} hit rate @1: {hit_rate(fn, 1):.0%}   @3: {hit_rate(fn, 3):.0%}")

### Re-ranking: a second, more careful look
Retrieval is fast but rough. A **cross-encoder** reads the question and each candidate chunk **together** (slower but more accurate), then re-orders the top results.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_search(question, k=3):
    candidates = hybrid_search(question, 10)
    scores = reranker.predict([(question, t) for t, _, _ in candidates])
    order = np.argsort(scores)[::-1][:k]
    return [(candidates[i][0], candidates[i][1], float(scores[i])) for i in order]

print(f"hybrid + re-rank hit rate @1: {hit_rate(rerank_search, 1):.0%}")

### 🧩 Exercise 2
1. Write 5 more evaluation questions — including some that use **different words** from the handbook. Which retrieval method wins now?
2. Swap `retrieve` for `rerank_search` inside `rag()`. Do the answers improve?
3. Hybrid search isn't automatically better on every dataset — on a tiny handbook, keyword search can drag it down. That's exactly why we **measure** instead of guessing.

## Part 5: The same RAG in LangChain 🔗
Frameworks package all these steps. Here's the whole pipeline in LangChain (Lesson 30).

In [ ]:
!pip -q install langchain langchain-ollama langchain-community langchain-huggingface langchain-text-splitters faiss-cpu
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

docs = DirectoryLoader("handbook", glob="*.md", loader_cls=TextLoader).load()                  # 1. load
splits = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50).split_documents(docs)   # 2. chunk
store = FAISS.from_documents(splits, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))        # 3–4. embed + store
retriever = store.as_retriever(search_kwargs={"k": 3})

def format_docs(ds):
    return "\n".join(f"[{i + 1}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(ds))

prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY these sources and cite them like [1]. If unsure, say you don't know.\n\n{context}\n\nQuestion: {question}")
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | prompt | ChatOllama(model=MODEL, temperature=0) | StrOutputParser())

print(rag_chain.invoke("How many books can I borrow and for how long?"))

### 🚀 Challenge
Make the RAG return **structured** output: `{"answer": ..., "sources": [...], "confident": true/false}`. Then show "please ask the office" whenever `confident` is false.

## 📌 Summary
- **RAG** = open-book exam for LLMs: retrieve relevant chunks, put them in the prompt, generate with citations
- **Ingestion**: load → chunk → embed → store. **Query**: embed → retrieve → augment → generate
- Quality depends on **chunking**, **k**, **retrieval method** (vector, keyword, hybrid) and **re-ranking**
- Always **evaluate** retrieval (hit rate) separately from answers